### 1. First cell — imports and project configuration
This uses your existing settings module rather than hardcoding project paths and hyperparameters.

In [6]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
import json
# Find project root
CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path for path in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (path / "config" / "settings.py").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find config/settings.py")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import settings

tf.random.set_seed(settings.RANDOM_STATE)
np.random.seed(settings.RANDOM_STATE)

IMAGE_SIZE = 299
IMAGE_CHANNELS = 3
BATCH_SIZE = settings.BATCH_SIZE
LEARNING_RATE = settings.LEARNING_RATE
EPOCHS = settings.EPOCHS

INPUT_SHAPE = (
    IMAGE_SIZE,
    IMAGE_SIZE,
    IMAGE_CHANNELS,
)

print("Project:", settings.PROJECT_NAME)
print("Input shape:", INPUT_SHAPE)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Configured epochs:", EPOCHS)

Project: BrainScan_AI
Input shape: (299, 299, 3)
Batch size: 32
Learning rate: 0.001
Configured epochs: 30


### 2 - Load and inspect dataset manifests

In [7]:
TRAIN_MANIFEST_PATH = settings.TRAIN_MANIFEST_PATH
VALIDATION_MANIFEST_PATH = settings.VALIDATION_MANIFEST_PATH

if not TRAIN_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Training manifest not found: {TRAIN_MANIFEST_PATH}\n"
        "Run Notebook 02 first."
    )

if not VALIDATION_MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f"Validation manifest not found: {VALIDATION_MANIFEST_PATH}\n"
        "Run Notebook 02 first."
    )

train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
validation_df = pd.read_csv(VALIDATION_MANIFEST_PATH)

print("Training shape: ", train_df.shape)
print("Validation shape: ", validation_df.shape)


print("\nTraining column")

print(train_df.columns.tolist())

display(train_df.head())

Training shape:  (4916, 9)
Validation shape:  (1053, 9)

Training column
['image_path', 'relative_path', 'label', 'extension', 'file_size_bytes', 'width', 'height', 'image_mode', 'is_valid']


,image_path,relative_path,label,extension,file_size_bytes,width,height,image_mode,is_valid
0,/home/moali/projects/brainscan_ai/data/raw/gli...,glioma/Te-gl_0274.jpg,glioma,.jpg,22680,512,512,L,True
1,/home/moali/projects/brainscan_ai/data/raw/not...,notumor/Tr-no_0984.jpg,notumor,.jpg,12060,236,236,RGB,True
2,/home/moali/projects/brainscan_ai/data/raw/men...,meningioma/Tr-me_1092.jpg,meningioma,.jpg,25839,512,512,L,True
3,/home/moali/projects/brainscan_ai/data/raw/gli...,glioma/Te-gl_0273.jpg,glioma,.jpg,23693,512,512,L,True
4,/home/moali/projects/brainscan_ai/data/raw/pit...,pituitary/Tr-pi_1247.jpg,pituitary,.jpg,26616,512,512,L,True


### 3 — Identify the classes

In [8]:
LABEL_COL = "label"  # Change this if your manifest uses another column

if LABEL_COL not in train_df.columns:
    raise ValueError(
        f"Column '{LABEL_COL}' not found. "
        f"Available columns: {train_df.columns.tolist()}"
    )

if LABEL_COL not in validation_df.columns:
    raise ValueError(
        f"Validation manifest is missing '{LABEL_COL}'."
    )

if train_df[LABEL_COL].isna().any():
    raise ValueError("Training labels contain missing values.")

if validation_df[LABEL_COL].isna().any():
    raise ValueError("Validation labels contain missing values.")

# Keep labels consistent, even if the CSV contains a mixture of strings/numbers
train_labels = train_df[LABEL_COL].astype(str)
validation_labels = validation_df[LABEL_COL].astype(str)

CLASS_NAMES = sorted(train_labels.unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)

unknown_validation_classes = (
    set(validation_labels.unique()) - set(CLASS_NAMES)
)

if unknown_validation_classes:
    raise ValueError(
        "Validation contains classes not found in training: "
        f"{unknown_validation_classes}"
    )

LABEL_TO_ID = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

print("Classes:", CLASS_NAMES)
print("Number of classes:", NUM_CLASSES)
print("Label mapping:", LABEL_TO_ID)

print("\nTraining class distribution:")
display(train_labels.value_counts().sort_index().to_frame("count"))

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Number of classes: 4
Label mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

Training class distribution:


,count
label,
glioma,1135
meningioma,1151
notumor,1400
pituitary,1230


### 4 — Implement the Inception module

In [9]:
def build_inceptionv3(num_classes: int) -> keras.Model:
    """
    Create an InceptionV3 transfer-learning classifier.
    The backbone starts with pretrained ImageNet weights and is frozen.
    """

    base_model = keras.applications.InceptionV3(
        include_top=False,
        weights="imagenet",
        input_shape=INPUT_SHAPE,
    )

    # Freeze pretrained feature-extraction layers initially
    base_model.trainable = False

    inputs = keras.Input(shape=INPUT_SHAPE, name="mri_image")

    # InceptionV3-specific preprocessing:
    # input pixels are expected in the [0, 255] range
    x = keras.applications.inception_v3.preprocess_input(inputs)

    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = layers.Dropout(0.4, name="dropout")(x)

    outputs = layers.Dense(
        num_classes,
        activation="softmax",
        name="predictions",
    )(x)

    model = keras.Model(
        inputs=inputs,
        outputs=outputs,
        name="BrainScan_InceptionV3",
    )

    return model

### 4.1 - model creation

In [10]:
model = build_inceptionv3(NUM_CLASSES)

model.summary()

print("Total parameters:", f"{model.count_params():,}")
print(
    "Trainable parameters:",
    f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}",
)

Model: "BrainScan_InceptionV3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mri_image (InputLayer)          │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 299, 299, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ inception_v3 (Functional)       │ (None, 8, 8, 2048)     │    21,802,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,810,980 (83.20 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 21,802,784 (83.17 MB)

Total parameters: 21,810,980
Trainable parameters: 8,196


### 5-Verify the forward pass

In [11]:

dummy_batch = tf.zeros((1, *INPUT_SHAPE), dtype=tf.float32)
predictions = model(dummy_batch, training=False)

assert predictions.shape == (1, NUM_CLASSES)
assert np.allclose(
    tf.reduce_sum(predictions, axis=-1).numpy(),
    1.0,
    atol=1e-5,
)

print("Input shape:", model.input_shape)
print("Output shape:", model.output_shape)
print("Forward-pass verification: passed")

Input shape: (None, 299, 299, 3)
Output shape: (None, 4)
Forward-pass verification: passed


### 6 - Compile the model

In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
    ],
)

print("InceptionV3 compiled successfully.")

InceptionV3 compiled successfully.


### 7 - Save the initial model and class mapping

In [13]:
MODEL_DIR = settings.CLASSIFICATION_MODELS_DIR
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "inceptionv3_initial.keras"
CLASS_MAPPING_PATH = MODEL_DIR / "inceptionv3_class_mapping.json"

model.save(MODEL_PATH)

with open(CLASS_MAPPING_PATH, "w", encoding="utf-8") as file:
    json.dump(
        {
            "architecture": "InceptionV3",
            "weights": "ImageNet",
            "input_shape": list(INPUT_SHAPE),
            "classes": CLASS_NAMES,
            "label_to_id": LABEL_TO_ID,
            "num_classes": NUM_CLASSES,
            "random_state": settings.RANDOM_STATE,
        },
        file,
        indent=4,
    )

print("Saved model:", MODEL_PATH)
print("Saved class mapping:", CLASS_MAPPING_PATH)

Saved model: /home/moali/projects/brainscan_ai/models/classification/inceptionv3_initial.keras
Saved class mapping: /home/moali/projects/brainscan_ai/models/classification/inceptionv3_class_mapping.json


### 8 - Final development summary

In [14]:
summary = {
    "architecture": "GoogLeNet (Inception v3)",
    "input_shape": list(INPUT_SHAPE),
    "classes": CLASS_NAMES,
    "number_of_classes": NUM_CLASSES,
    "total_parameters": int(model.count_params()),
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "loss": "SparseCategoricalCrossentropy",
    "training_performed": False,
}

display(pd.DataFrame(
    [{"Property": key, "Value": str(value)}
     for key, value in summary.items()]
))

,Property,Value
0,architecture,GoogLeNet (Inception v3)
1,input_shape,"[299, 299, 3]"
2,classes,"['glioma', 'meningioma', 'notumor', 'pituitary']"
3,number_of_classes,4
4,total_parameters,21810980
5,optimizer,Adam
6,learning_rate,0.001
7,loss,SparseCategoricalCrossentropy
8,training_performed,False
